# Writing a Constitution [Step 04.02]

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 ended with an ad-hoc list of five rules that worked better than "is
this good?". Formalising that list is **Constitutional AI** (Bai et al., 2022):

> A written set of principles the model critiques its own output against, then
> revises to satisfy.

```
   CONSTITUTION                 draft
   - principle 1                  |
   - principle 2   ------------>  critique against EACH principle
   - principle 3                  |
   ...                            v
                                revise
```

The constitution is an **artifact**: version-controlled, reviewable by
non-engineers, and testable. That is its real advantage over a long system
prompt, and it is more of an organisational advantage than a technical one.

### What you'll learn

- What makes a principle usable by a model versus merely well-intentioned.
- The four failure modes of badly written principles.
- How to test a principle before you trust it.
- Why principles need a stated **priority order**.

### Why it matters

Anyone can write "be helpful and harmless". The engineering is in writing
principles that are specific enough to be checkable, narrow enough not to fire on
everything, and ordered so the model knows what to do when two of them conflict -
which they will.

### Prerequisites

- [01_the_critique_loop](01_the_critique_loop.ipynb)
- [05_production_security/02_guardrail_frameworks](../../../05_production_security) - guardrails, which enforce rules from outside rather than from within.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### The shared task


In [ ]:
# One drafting task, used by every notebook in this module so the comparisons are
# apples to apples. It is chosen because it has MANY ways to go subtly wrong -
# which is exactly the situation principles are for.

SITUATION = """A customer, Elena Duarte, has emailed angrily. Her order NW-10261
(SKU TH-275-BLK, supplied by Meridian Components GmbH) was cancelled by us
without warning because the frame size was discontinued. She has been waiting
eleven days. Our records show she was charged and NOT yet refunded; finance says
the refund will clear in 3-5 business days but has occasionally taken longer.
She has asked for compensation. Our policy allows a goodwill voucher of up to
15 EUR, which requires a supervisor's approval that has not yet been given."""

DRAFT_INSTRUCTION = ("Write the reply we should send to Elena. Write only the "
                     "email body, no subject line and no notes.")

print(SITUATION)


### Deterministic checks


In [ ]:
# Not every principle can be checked by code, but several can - and the ones that
# can are worth far more than an LLM's opinion, because they cannot be argued
# with. We use them to MEASURE whether critique-and-revise actually changed
# anything, rather than trusting the model's report of its own improvement.

import re

INTERNAL_TERMS = ["th-275", "meridian", "supplier", "sku", "finance says",
                  "our records show"]
OVERPROMISE = ["guarantee", "guaranteed", "i promise", "we promise", "definitely will",
               "will certainly", "rest assured that you will", "immediately refund"]
EMPATHY = ["sorry", "apolog", "understand", "frustrat", "disappoint", "regret"]


def check(text):
    """Return a dict of objective observations about a draft."""
    low = text.lower()
    words = len(re.findall(r"\b[\w'-]+\b", text))
    return {
        "words": words,
        "over_150_words": words > 150,
        "internal_leaks": [t for t in INTERNAL_TERMS if t in low],
        "overpromises": [t for t in OVERPROMISE if t in low],
        "has_empathy": any(t in low for t in EMPATHY),
        "mentions_voucher": "voucher" in low or "15 eur" in low or "15EUR".lower() in low,
        "states_refund_window": bool(re.search(r"3\s*[-to]+\s*5\s*business days", low)),
    }


def violations(text):
    """Count objective violations. Lower is better."""
    c = check(text)
    n = 0
    n += len(c["internal_leaks"])
    n += len(c["overpromises"])
    n += int(c["over_150_words"])
    n += int(not c["has_empathy"])
    return n


def report(text, label):
    c = check(text)
    print("%-22s words=%-4d leaks=%-22s overpromise=%-18s empathy=%-5s -> %d violations"
          % (label, c["words"], ",".join(c["internal_leaks"]) or "none",
             ",".join(c["overpromises"]) or "none", c["has_empathy"], violations(text)))
    return c


### 1. What a usable principle looks like

Four properties. A principle missing any one of them will not change behaviour:

| Property | Bad | Good |
|---|---|---|
| **Specific** | "Be professional" | "Do not name suppliers, SKUs or internal systems" |
| **Checkable** | "Be helpful" | "State the next concrete step the customer should expect" |
| **Narrow** | "Be safe" | "Do not commit to a date the company does not control" |
| **Actionable** | "Avoid problems" | "If compensation is not yet approved, do not mention an amount" |

The test that catches most bad principles: **can you tell, from the text alone,
whether a given draft violates it?** If two reasonable people would disagree, the
model will disagree with itself too.

In [4]:
CONSTITUTION = [
    dict(id="P1", name="No internal information",
         rule="The reply must not disclose supplier names, SKU or part codes, "
              "internal system names, or what internal departments have said.",
         fix="Remove the internal detail entirely. Do not replace it with a "
             "vaguer version of the same information."),
    dict(id="P2", name="No uncontrollable commitments",
         rule="The reply must not promise a specific date, guarantee an outcome, "
              "or use words like 'guarantee', 'definitely' or 'I promise' about "
              "anything the company does not directly control.",
         fix="State what has been done and what happens next, with the company's "
             "normal timescale described as typical rather than promised."),
    dict(id="P3", name="No unapproved compensation",
         rule="The reply must not offer, imply or quote a compensation amount "
              "unless approval has already been given.",
         fix="Say that a goodwill gesture is being reviewed, without an amount."),
    dict(id="P4", name="Acknowledge the impact",
         rule="The reply must explicitly acknowledge the customer's frustration "
              "and the time they have waited, before explaining anything.",
         fix="Open with a direct acknowledgement of the delay and its effect."),
    dict(id="P5", name="Give one concrete next step",
         rule="The reply must tell the customer exactly one thing that will "
              "happen next, and who will do it.",
         fix="Add a single sentence naming the next action and its owner."),
    dict(id="P6", name="Brevity",
         rule="The reply must be under 150 words.",
         fix="Cut explanation, not acknowledgement. Remove background the "
             "customer did not ask for."),
]

for p in CONSTITUTION:
    print("%s  %s\n    %s\n" % (p["id"], p["name"], p["rule"]))

P1  No internal information
    The reply must not disclose supplier names, SKU or part codes, internal system names, or what internal departments have said.

P2  No uncontrollable commitments
    The reply must not promise a specific date, guarantee an outcome, or use words like 'guarantee', 'definitely' or 'I promise' about anything the company does not directly control.

P3  No unapproved compensation
    The reply must not offer, imply or quote a compensation amount unless approval has already been given.

P4  Acknowledge the impact
    The reply must explicitly acknowledge the customer's frustration and the time they have waited, before explaining anything.

P5  Give one concrete next step
    The reply must tell the customer exactly one thing that will happen next, and who will do it.

P6  Brevity
    The reply must be under 150 words.



> **Note what each principle carries:** a `rule` (how to detect a violation) and
> a `fix` (what to do about it). Most published examples give only the rule, and
> then the revision drifts - the model knows something is wrong but invents its
> own remedy. Pairing each rule with its remedy is the single biggest practical
> improvement you can make to a constitution.

### 2. Priority order - principles conflict

P4 (acknowledge the impact) and P6 (under 150 words) pull against each other.
So do P3 (no unapproved compensation) and any principle about making the customer
feel compensated.

An unordered constitution leaves the model to resolve those silently and
inconsistently. So state the order, and state it as a rule of its own.

In [5]:
PRIORITY = """When two principles conflict, apply them in this order, highest first:

1. P1, P2, P3 - the SAFETY principles. Never violate these to satisfy anything else.
2. P4 - acknowledgement. The customer must feel heard.
3. P5 - the next step.
4. P6 - brevity. Sacrifice brevity before sacrificing anything above it.

Never resolve a conflict by partially satisfying a safety principle."""

print(PRIORITY)

When two principles conflict, apply them in this order, highest first:

1. P1, P2, P3 - the SAFETY principles. Never violate these to satisfy anything else.
2. P4 - acknowledgement. The customer must feel heard.
3. P5 - the next step.
4. P6 - brevity. Sacrifice brevity before sacrificing anything above it.

Never resolve a conflict by partially satisfying a safety principle.


The general rule, which transfers to any constitution you write:

```
   safety / legal   >   truthfulness   >   helpfulness   >   style
```

Brevity is style. It goes last, always. Teams get this wrong because brevity is
the easiest property to measure, so it ends up over-weighted.

### 3. Testing a principle before you trust it

A principle is a piece of software. Test it: write a draft that obviously
violates it and one that obviously does not, then check the critic gets both
right.

We test P1 here, on inputs where we already know the answer.

In [6]:
CRITIC_ONE = """You are checking a customer service reply against ONE principle.

PRINCIPLE {pid} - {name}: {rule}

Reply with exactly one line:
VIOLATION: <the exact offending text>
or
OK

REPLY TO CHECK:
{text}"""


def check_principle(principle, text):
    out = chat([("user", CRITIC_ONE.format(pid=principle["id"], name=principle["name"],
                                           rule=principle["rule"], text=text))],
               temperature=0.0, max_tokens=110).content.strip()
    return out.upper().startswith("VIOLATION"), out


P1 = CONSTITUTION[0]

SHOULD_FAIL = ("Hi Elena, your frame (SKU TH-275-BLK) was discontinued by our "
               "supplier Meridian Components GmbH, so we cancelled the order.")
SHOULD_PASS = ("Hi Elena, the frame you ordered is no longer available, so we had "
               "to cancel the order. I'm sorry we did not tell you sooner.")

for label, text, expected in (("obvious violation", SHOULD_FAIL, True),
                              ("clean version", SHOULD_PASS, False)):
    flagged, out = check_principle(P1, text)
    print("%-18s expected_violation=%-5s got=%-5s  %s"
          % (label, expected, flagged, "PASS" if flagged == expected else "*** FAIL ***"))
    print("    critic said: %s\n" % out.replace("\n", " ")[:90])

obvious violation  expected_violation=True  got=True   PASS
    critic said: VIOLATION: SKU TH-275-BLK, Meridian Components GmbH



clean version      expected_violation=False got=False  PASS
    critic said: OK



If the second case is flagged, your principle is **over-broad** - it will fire on
clean text and the revision loop will damage good drafts trying to fix
non-problems. If the first is missed, it is **under-specified**. Either way, edit
the principle, not the model.

### 4. The four failure modes

| Failure | Symptom | Fix |
|---|---|---|
| **Too vague** | Critic says OK to everything | Name the exact thing that is forbidden |
| **Too broad** | Critic flags clean drafts | Add the boundary: what is explicitly allowed |
| **Overlapping** | Two principles fire on the same text | Merge them, or make one clearly narrower |
| **Unresolvable** | Principle demands something the situation forbids | Make it conditional |

The third is the sneakiest. Overlapping principles make revision unstable: fixing
P2 re-triggers P3, whose fix re-triggers P2, and the loop oscillates. Let us look
for overlap in ours.

### Does more than one principle fire on the same offending text?


In [ ]:
BAD_DRAFT = ("Hi Elena, I'm so sorry. Finance says your refund from supplier "
             "Meridian will definitely clear on Tuesday, and I guarantee you a "
             "15 EUR voucher for the trouble.")

fired = []
for p in CONSTITUTION[:3]:
    flagged, out = check_principle(p, BAD_DRAFT)
    fired.append((p["id"], flagged, out.replace("\n", " ")[:64]))
    print("%s %-28s %-5s %s" % (p["id"], p["name"], flagged, out.replace("\n", " ")[:56]))

n = sum(1 for _, f, _ in fired if f)
print("\n%d of 3 safety principles fired on this draft." % n)
print("Multiple principles firing on DIFFERENT text is correct and expected.")
print("Multiple principles firing on the SAME quoted text means they overlap -")
print("check the quotes above, not just the flags.")


### 5. Constitution vs. system prompt

A fair question: why not just put all of this in the system prompt?

| | System prompt | Constitution |
|---|---|---|
| **When it applies** | Before generation | After generation, as a check |
| **Cost** | Every call, forever | Only on the calls you choose to critique |
| **Failure mode** | Silently ignored under load | Produces an explicit violation you can log |
| **Reviewable by** | Whoever edits the prompt | Legal, support leads, compliance |
| **Testable** | Hard | Each principle, individually |

The honest answer is: **do both.** Put the principles in the system prompt so
most drafts are clean, and run the constitution as a check so the ones that are
not get caught. The constitution's unique value is that a violation becomes an
*event* - loggable, countable, alertable - instead of an absence.

In [8]:
# The system-prompt half. Cheap, and it prevents most violations up front.
SYSTEM_FROM_CONSTITUTION = ("You write customer service replies. Follow these rules:\n"
                            + "\n".join("- %s" % p["rule"] for p in CONSTITUTION))

prompted = chat([("system", SYSTEM_FROM_CONSTITUTION),
                 ("user", SITUATION + "\n\n" + DRAFT_INSTRUCTION)],
                temperature=0.3, max_tokens=380).content.strip()
plain = chat([("user", SITUATION + "\n\n" + DRAFT_INSTRUCTION)],
             temperature=0.3, max_tokens=380).content.strip()

print(prompted)
print()
_ = report(plain, "no constitution")
_ = report(prompted, "constitution in prompt")
print()
print("Putting the principles in the prompt is the cheapest win available.")
print("It does not remove the need to CHECK - it reduces how often the check fires.")

Dear Elena,

I am truly sorry for the frustration and the eleven days you have had to wait without clear communication regarding your order. I completely understand how unacceptable this silence and the delay have been.

Your order was cancelled because the specific frame size is no longer available. I can confirm that your refund is currently being processed and will be issued to your original payment method. While this typically takes a few business days to appear, please allow up to five business days for it to clear.

Regarding your request for compensation, I have escalated this to our supervisor for review. You will receive a separate email from our team within 48 hours confirming the outcome of that review.

Thank you for your patience while we resolve this.

Sincerely,

Customer Support Team

no constitution        words=238  leaks=th-275,meridian,supplier,sku overpromise=none               empathy=True  -> 5 violations
constitution in prompt words=134  leaks=none              

### 6. Pitfalls

- **Aspirational principles.** "Be empathetic" is a value, not a rule. Write the
  behaviour that would demonstrate it.
- **No priority order.** Conflicts get resolved silently and differently each time.
- **Untested principles.** A principle that fires on everything is worse than no
  principle, because it damages good drafts.
- **Too many.** Six is workable. Twenty means each gets a fraction of the model's
  attention, and several will overlap.
- **Rules without remedies.** Pair each rule with what to do about it.

### Recap

| Idea | Takeaway |
|---|---|
| Specific, checkable, narrow, actionable | The four properties of a usable principle |
| Rule + fix | Pair every rule with its remedy or revision drifts |
| Priority order | Safety > truth > helpfulness > style. Always. |
| Test each principle | Two inputs where you know the answer |
| Prompt *and* check | Prompting prevents; checking makes violations countable |

**Next:** [03_constitutional_loop](03_constitutional_loop.ipynb) - running the full
critique-and-revise loop against this constitution, with measured violations.